"""
statistical_analysis.py
Roadmap item 7: comprehensive statistical validation of MSGR-RAG results.

Extends the original bootstrap CI + McNemar's test script with:
  1. Bootstrap confidence intervals (overall + per-category)      [already present]
  2. McNemar's test — MSGR-RAG vs. Semantic RAG (paired)           [already present]
  3. Cohen's h — effect size for the McNemar's comparison          [NEW]
  4. Mann-Whitney U test — score separation (S_E, S_C, S_D, C_f,
     Conf_f) between verified and hallucinated facts               [NEW]
  5. ROC curve + AUC — threshold-independent discrimination using
     Conf(f) as a continuous score against ground truth            [NEW]

All analyses run against verification_results.csv (974 facts, post
test-set cleanup) and semantic_rag_predictions.csv (Semantic RAG
baseline), aligned by exact (subject, predicate, object) match.
"""

In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy.stats import chi2, mannwhitneyu

INTERMEDIATE_DIR = Path("../Datasets/intermediate")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

N_BOOTSTRAP = 1000
SEED = 42
BEST_TC, BEST_TF = 0.50, 0.55

np.random.seed(SEED)


In [3]:
# ============================================================================
# Shared helpers
# ============================================================================

def recompute_decision(row, tc, tf):
    """Re-apply the three-level hierarchical decision rule at given thresholds."""
    if row['C_f'] < tc:
        return 'HALLUCINATION'
    if row['patterns_detected'] != 'None' and pd.notna(row['patterns_detected']):
        return 'HALLUCINATION'
    if row['Conf_f'] < tf:
        return 'HALLUCINATION'
    return 'VERIFIED'


def compute_metrics(df, decision_col='decision'):
    TP = len(df[(df.ground_truth == 'HALLUCINATION') & (df[decision_col] == 'HALLUCINATION')])
    TN = len(df[(df.ground_truth == 'VERIFIED') & (df[decision_col] == 'VERIFIED')])
    FP = len(df[(df.ground_truth == 'VERIFIED') & (df[decision_col] == 'HALLUCINATION')])
    FN = len(df[(df.ground_truth == 'HALLUCINATION') & (df[decision_col] == 'VERIFIED')])
    total = len(df)
    accuracy = (TP + TN) / total if total else 0
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}


In [4]:
# ============================================================================
# 1. Bootstrap confidence intervals
# ============================================================================

def bootstrap_ci(df, decision_col='decision', n_bootstrap=N_BOOTSTRAP, category=None):
    """Bootstrap 95% CI for accuracy/precision/recall/F1."""
    if category:
        df = df[df['category'] == category]
    n = len(df)
    if n == 0:
        return None, 0

    metrics_samples = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}
    for _ in range(n_bootstrap):
        sample = df.sample(n=n, replace=True)
        m = compute_metrics(sample, decision_col)
        for k in metrics_samples:
            metrics_samples[k].append(m[k])

    result = {}
    point = compute_metrics(df, decision_col)
    for k, vals in metrics_samples.items():
        vals = np.array(vals)
        result[k] = {
            'point_estimate': point[k],
            'ci_lower': float(np.percentile(vals, 2.5)),
            'ci_upper': float(np.percentile(vals, 97.5)),
        }
    return result, n

In [5]:
# ============================================================================
# 2. McNemar's test + 3. Cohen's h (effect size)
# ============================================================================

def mcnemar_test(msgr_correct, semantic_correct):
    """Continuity-corrected McNemar's test for paired binary classification."""
    b = int(np.sum(msgr_correct & ~semantic_correct))
    c = int(np.sum(~msgr_correct & semantic_correct))

    if b + c == 0:
        return {'b': b, 'c': c, 'statistic': 0.0, 'p_value': 1.0}

    statistic = (abs(b - c) - 1) ** 2 / (b + c)
    p_value = float(1 - chi2.cdf(statistic, df=1))
    return {'b': b, 'c': c, 'statistic': float(statistic), 'p_value': p_value}


def cohens_h(p1, p2):
    """
    Cohen's h — effect size for the difference between two proportions.
    Conventional interpretation: 0.2 small, 0.5 medium, 0.8 large.
    """
    phi1 = 2 * np.arcsin(np.sqrt(p1))
    phi2 = 2 * np.arcsin(np.sqrt(p2))
    h = abs(phi1 - phi2)
    if h < 0.2:
        magnitude = 'negligible'
    elif h < 0.5:
        magnitude = 'small'
    elif h < 0.8:
        magnitude = 'medium'
    else:
        magnitude = 'large'
    return {'h': float(h), 'magnitude': magnitude}

In [6]:
# ============================================================================
# 4. Mann-Whitney U — score separation between verified and hallucinated
# ============================================================================

def mann_whitney_scores(results_df):
    """
    Non-parametric test confirming each verification scale (S_E, S_C, S_D)
    and the aggregate scores (C_f, Conf_f) meaningfully separate verified
    from hallucinated facts — not just different means, but a statistically
    significant distributional difference.
    """
    verified = results_df[results_df.ground_truth == 'VERIFIED']
    hallucinated = results_df[results_df.ground_truth == 'HALLUCINATION']

    scores_to_test = ['S_E', 'S_C', 'S_D', 'C_f', 'Conf_f']
    results = {}

    for score in scores_to_test:
        stat, p = mannwhitneyu(
            verified[score].dropna(),
            hallucinated[score].dropna(),
            alternative='two-sided'
        )
        # Rank-biserial correlation as effect size for Mann-Whitney
        n1, n2 = len(verified[score].dropna()), len(hallucinated[score].dropna())
        rank_biserial = 1 - (2 * stat) / (n1 * n2)

        results[score] = {
            'verified_median': float(verified[score].median()),
            'hallucinated_median': float(hallucinated[score].median()),
            'u_statistic': float(stat),
            'p_value': float(p),
            'rank_biserial_effect_size': float(rank_biserial),
            'significant_p001': bool(p < 0.001),
        }

    return results

In [7]:
# ============================================================================
# 5. ROC / AUC — threshold-independent discrimination
# ============================================================================

def roc_auc_analysis(results_df):
    """
    Compute ROC curve and AUC using Conf(f) as a continuous score against
    ground truth, independent of the specific theta_confidence threshold
    chosen for the headline results. Positive class = HALLUCINATION
    (i.e. higher Conf(f) should predict VERIFIED, so we invert for scoring).
    """
    from sklearn.metrics import roc_curve, auc

    y_true = (results_df['ground_truth'] == 'HALLUCINATION').astype(int)
    # Use (1 - Conf_f) as the "hallucination score" — higher Conf_f means more
    # likely VERIFIED, so 1 - Conf_f is the probability-like score for HALLUCINATION
    y_score = 1 - results_df['Conf_f']

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    # Also compute AUC using C_f alone, and combined C_f+Conf_f average, for comparison
    y_score_cf = 1 - results_df['C_f']
    fpr_cf, tpr_cf, _ = roc_curve(y_true, y_score_cf)
    auc_cf = auc(fpr_cf, tpr_cf)

    return {
        'auc_conf_f': float(roc_auc),
        'auc_c_f': float(auc_cf),
        'fpr': fpr.tolist(),
        'tpr': tpr.tolist(),
        'thresholds': thresholds.tolist(),
    }


def plot_roc_curve(roc_data, output_path):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(roc_data['fpr'], roc_data['tpr'], color='#2E86AB', linewidth=2,
            label=f"Conf(f)  (AUC = {roc_data['auc_conf_f']:.3f})")
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, label='Chance (AUC = 0.500)')
    ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
    ax.set_title('ROC Curve: Hallucination Detection via Conf(f)\n(threshold-independent)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_path}")

In [8]:
# ============================================================================
# Main
# ============================================================================

def main():
    results_df = pd.read_csv(INTERMEDIATE_DIR / "verification_results.csv")
    results_df['decision'] = results_df.apply(lambda r: recompute_decision(r, BEST_TC, BEST_TF), axis=1)

    print("=" * 70)
    print("1. BOOTSTRAP CONFIDENCE INTERVALS — MSGR-RAG (overall)")
    print("=" * 70)
    overall_ci, n = bootstrap_ci(results_df)
    for metric, vals in overall_ci.items():
        print(f"{metric:12s}: {vals['point_estimate']*100:5.1f}%  "
              f"(95% CI: {vals['ci_lower']*100:5.1f}% - {vals['ci_upper']*100:5.1f}%)  [n={n}]")

    print("\n" + "=" * 70)
    print("BOOTSTRAP CONFIDENCE INTERVALS — per category")
    print("=" * 70)
    category_ci = {}
    for cat in results_df['category'].unique():
        cat_ci, n_cat = bootstrap_ci(results_df, category=cat)
        category_ci[cat] = cat_ci
        acc = cat_ci['accuracy']
        print(f"{cat:28s} (n={n_cat:4d}): {acc['point_estimate']*100:5.1f}%  "
              f"(95% CI: {acc['ci_lower']*100:5.1f}% - {acc['ci_upper']*100:5.1f}%)")

    print("\n" + "=" * 70)
    print("2. MCNEMAR'S TEST + 3. COHEN'S h — MSGR-RAG vs. Semantic RAG")
    print("=" * 70)

    semantic_preds = pd.read_csv(INTERMEDIATE_DIR / "semantic_rag_predictions.csv")
    merged = results_df.merge(
        semantic_preds[['subject', 'predicate', 'object', 'decision']],
        on=['subject', 'predicate', 'object'],
        suffixes=('_msgr', '_semantic')
    )
    print(f"Aligned fact count for paired comparison: {len(merged)}")

    msgr_correct = (merged['decision_msgr'] == merged['ground_truth']).values
    semantic_correct = (merged['decision_semantic'] == merged['ground_truth']).values

    mcnemar_result = mcnemar_test(msgr_correct, semantic_correct)
    print(f"\nDiscordant pairs: b(MSGR correct, Semantic wrong)={mcnemar_result['b']}, "
          f"c(MSGR wrong, Semantic correct)={mcnemar_result['c']}")
    print(f"McNemar chi-square: {mcnemar_result['statistic']:.4f}, p-value: {mcnemar_result['p_value']:.6f}")

    p_msgr = msgr_correct.mean()
    p_semantic = semantic_correct.mean()
    effect = cohens_h(p_msgr, p_semantic)
    print(f"\nAccuracy: MSGR-RAG={p_msgr*100:.1f}%, Semantic RAG={p_semantic*100:.1f}%")
    print(f"Cohen's h: {effect['h']:.3f} ({effect['magnitude']} effect)")

    print("\n" + "=" * 70)
    print("4. MANN-WHITNEY U TEST — score separation (verified vs. hallucinated)")
    print("=" * 70)
    mw_results = mann_whitney_scores(results_df)
    for score, r in mw_results.items():
        sig = "***" if r['significant_p001'] else ("*" if r['p_value'] < 0.05 else "ns")
        print(f"{score:8s}: median(VERIF)={r['verified_median']:.3f}  "
              f"median(HALL)={r['hallucinated_median']:.3f}  "
              f"p={r['p_value']:.2e} {sig}  effect_size(r)={r['rank_biserial_effect_size']:.3f}")

    print("\n" + "=" * 70)
    print("5. ROC / AUC ANALYSIS — threshold-independent discrimination")
    print("=" * 70)
    roc_data = roc_auc_analysis(results_df)
    print(f"AUC using Conf(f): {roc_data['auc_conf_f']:.4f}")
    print(f"AUC using C_f alone: {roc_data['auc_c_f']:.4f}")
    plot_roc_curve(roc_data, FIGURES_DIR / "fig5_roc_curve.png")

    # ========================================================================
    # Save everything
    # ========================================================================
    output = {
        'overall_bootstrap_ci': overall_ci,
        'category_bootstrap_ci': category_ci,
        'mcnemar_test': mcnemar_result,
        'cohens_h': effect,
        'mann_whitney_scores': mw_results,
        'roc_auc': {'auc_conf_f': roc_data['auc_conf_f'], 'auc_c_f': roc_data['auc_c_f']},
        'n_bootstrap': N_BOOTSTRAP,
    }
    with open(INTERMEDIATE_DIR / "statistical_analysis_report.json", 'w') as f:
        json.dump(output, f, indent=2)
    print(f"\nSaved: {INTERMEDIATE_DIR / 'statistical_analysis_report.json'}")

In [9]:
main()

1. BOOTSTRAP CONFIDENCE INTERVALS — MSGR-RAG (overall)
accuracy    :  91.3%  (95% CI:  89.4% -  93.1%)  [n=974]
precision   :  87.1%  (95% CI:  84.5% -  89.7%)  [n=974]
recall      : 100.0%  (95% CI: 100.0% - 100.0%)  [n=974]
f1          :  93.1%  (95% CI:  91.6% -  94.6%)  [n=974]

BOOTSTRAP CONFIDENCE INTERVALS — per category
nonexistent_relationship     (n= 299): 100.0%  (95% CI: 100.0% - 100.0%)
real_edge                    (n= 400):  78.8%  (95% CI:  74.8% -  82.8%)
domain_violation             (n=  75): 100.0%  (95% CI: 100.0% - 100.0%)
wrong_predicate              (n= 200): 100.0%  (95% CI: 100.0% - 100.0%)

2. MCNEMAR'S TEST + 3. COHEN'S h — MSGR-RAG vs. Semantic RAG
Aligned fact count for paired comparison: 914

Discordant pairs: b(MSGR correct, Semantic wrong)=287, c(MSGR wrong, Semantic correct)=85
McNemar chi-square: 108.6048, p-value: 0.000000

Accuracy: MSGR-RAG=90.7%, Semantic RAG=68.6%
Cohen's h: 0.570 (medium effect)

4. MANN-WHITNEY U TEST — score separation (verified

In [10]:
import pandas as pd
results_df = pd.read_csv("../Datasets/intermediate/verification_results.csv")

verified_conf = results_df[results_df.ground_truth == 'VERIFIED']['Conf_f']
hall_conf = results_df[results_df.ground_truth == 'HALLUCINATION']['Conf_f']

print(f"Verified Conf_f range: [{verified_conf.min():.4f}, {verified_conf.max():.4f}]")
print(f"Hallucinated Conf_f range: [{hall_conf.min():.4f}, {hall_conf.max():.4f}]")
print(f"\nAny overlap? verified_min > hallucinated_max: {verified_conf.min() > hall_conf.max()}")

Verified Conf_f range: [0.7220, 1.0000]
Hallucinated Conf_f range: [0.3455, 0.6520]

Any overlap? verified_min > hallucinated_max: True


In [11]:
import pandas as pd

results_df = pd.read_csv("../Datasets/intermediate/verification_results.csv")

def recompute_decision_with_level(row, tc=0.5, tf=0.55):
    if row['C_f'] < tc:
        return 'HALLUCINATION', 1
    if row['patterns_detected'] != 'None' and pd.notna(row['patterns_detected']):
        return 'HALLUCINATION', 2
    if row['Conf_f'] < tf:
        return 'HALLUCINATION', 3
    return 'VERIFIED', None

results_df[['decision', 'level']] = results_df.apply(
    lambda r: pd.Series(recompute_decision_with_level(r)), axis=1
)

fps = results_df[(results_df.ground_truth == 'VERIFIED') & (results_df.decision == 'HALLUCINATION')]
print(f"Total FPs: {len(fps)}")
print(fps['level'].value_counts())
print(f"\nOf these FPs, how many have Conf_f > 0.55 (i.e. would pass Level 3 if reached)?")
print((fps['Conf_f'] > 0.55).sum(), "/", len(fps))

Total FPs: 85
level
1.0    78
2.0     7
Name: count, dtype: int64

Of these FPs, how many have Conf_f > 0.55 (i.e. would pass Level 3 if reached)?
85 / 85
